<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 40px; border-radius: 12px; border: 1px solid #30363d; text-align: center; color: white;">
  <span style="background: rgba(255,255,255,0.2); border: 1px solid rgba(255,255,255,0.4); color: white; padding: 4px 14px; border-radius: 20px; font-size: 12px; font-weight: 600; text-transform: uppercase;">Kafka Training · Lab 9</span>
  <h1 style="color: #ffffff; font-size: 2.4em; font-weight: bold; margin-top: 15px;">Schema Registry & Avro Serialization</h1>
  <p style="color: #e0e0e0; font-size: 1.1em;">Learn how to manage schemas, serialize data with Avro, and handle schema evolution in Kafka.</p>
</div>

---

## 🎯 Overview

**Confluent Schema Registry** is a distributed system for managing and versioning Avro schemas. It provides a centralized repository for schema management and enables schema evolution while maintaining compatibility.

**Why Schema Registry?**
- **Data Validation**: Ensures messages conform to defined schemas
- **Schema Evolution**: Handle changes to data structures safely
- **Compatibility Checking**: Prevent incompatible schema changes
- **Language Agnostic**: Use schemas across different programming languages
- **Subject Naming Strategies**: Organize schemas by topic or record type

**Avro Benefits:**
- Compact binary format (smaller message size)
- Strong typing and schema validation
- Language-independent serialization
- Support for complex nested types
- Automatic schema versioning

---

## ⚙️ Prerequisites

<div style="background-color: rgba(243, 156, 18, 0.1); border-left: 4px solid #f39c12; padding: 10px 15px; margin: 15px 0; border-radius: 4px;">
  <strong>⚠️ Important:</strong> We will use the docker-compose.yml from the Labs directory which includes Kafka, Zookeeper, and Schema Registry services.
</div>

---

## <span style="color: #667eea;">Step 1:</span> Start the Kafka Cluster with Schema Registry

We'll start a complete Kafka infrastructure including:
- **Zookeeper**: Coordination and leader election
- **Kafka Broker**: Message storage and distribution
- **Confluent Schema Registry**: Schema management service (runs on port 8081)


In [ ]:
!docker-compose -f ../../docker-compose.yml up -d

---

## <span style="color: #667eea;">Step 2:</span> Create Topic for Avro Messages

Create a topic that will store messages serialized with Avro schemas. This topic will automatically store schema IDs in message headers.


In [ ]:
!docker exec kafka kafka-topics \
  --bootstrap-server localhost:9092 \
  --create \
  --topic avro-topic \
  --partitions 1 \
  --replication-factor 1 \
  --if-not-exists

print("✓ Topic 'avro-topic' created!")

---

---

## <span style="color: #667eea;">Step 4:</span> Review External Configuration Files

**Externalized Files:**

| File | Purpose | Format |
|------|---------|--------|
| `order_schema_v1.avsc` | Avro schema definition | AVSC (Avro Schema) |
| `orders.csv` | Sample order data | CSV |


**What is .avsc?**
- **AVSC** = Avro Schema Compact format
- Standard file extension for Avro schemas
- Contains the same Avro schema definition that gets serialized into binary format
- More semantically correct than `.json` for Avro schemas

Open and review the configuration files:


In [ ]:
%pip install confluent-kafka pandas fastavro
print("✓ Required packages installed!")

---

## <span style="color: #667eea;">Step 6:</span> Run Avro Producer

`producer.py` script now:
1. **Loads schema** from `order_schema_v1.avsc` file
2. **Loads data** from `orders.csv` file (instead of hardcoded in code)
3. Connects to Schema Registry
4. Automatically registers the schema
5. Serializes order data using the registered schema
6. Produces messages to the `avro-topic`

**Code Flow:**
```
producer.py
  ├─ load_schema('order_schema_v1.avsc')  → AVSC file
  ├─ load_data('orders.csv')              → CSV file
  └─ Produce messages with external data
```

**Key Points:**
- Schema file is in `.avsc` format (Avro Schema standard)
- The Avro schema gets registered in Schema Registry
- Each message includes the schema ID for deserialization
- Code is completely decoupled from schema/data

Run the producer:


In [42]:
!python producer.py

✓ Schema loaded from order_schema_v1.avsc
✓ Data loaded from orders.csv (5 records)

Producing 5 messages...
------------------------------------------------------------
Producing: ORD001 - Alice Johnson
Producing: ORD002 - Bob Smith
Producing: ORD003 - Carol Davis
Producing: ORD004 - David Wilson
Producing: ORD005 - Eve Thompson
------------------------------------------------------------
Flushing messages to Kafka...

PRODUCTION SUMMARY
✓ Messages delivered: 0/5
✓ Schema registered in Schema Registry
❌ FAILURE: No messages were delivered!


---

## <span style="color: #667eea;">Step 7:</span> Query Schema Registry Using cURL Commands

Use curl to query the Schema Registry REST API. Here are the most common operations you'll need.

### 📋 Query 1: Health Check

Check if Schema Registry is running and responding on port 8081.

In [ ]:
!curl -s http://localhost:8081/

### 📋 Query 2: List All Registered Subjects

Get a list of all subjects (schemas) registered in Schema Registry. After the producer runs, you should see `"avro-topic-value"`.

In [ ]:
!curl -s http://localhost:8081/subjects

### 📋 Query 3: List All Versions of a Subject

Get all schema versions registered for the `avro-topic-value` subject. This shows the version history of the schema as it evolved.

In [ ]:
!curl -s http://localhost:8081/subjects/avro-topic-value/versions

### 📋 Query 4: Get Latest Schema Version

Get the latest version of the schema for the `avro-topic-value` subject, including the full schema definition and schema ID.

In [ ]:
!curl -s http://localhost:8081/subjects/avro-topic-value/versions/latest

### 📋 Query 5: Get Schema by Schema ID

Retrieve a specific schema using its ID. The schema ID is embedded in each Avro message and allows consumers to deserialize messages correctly. Replace `{schema_id}` with the actual ID from the previous query result.

In [ ]:
!curl -s http://localhost:8081/schemas/ids/1

---

## <span style="color: #667eea;">Step 8:</span> Consume Avro Messages with Automatic Deserialization

Run the consumer to read the Avro-encoded messages. The refactored `consumer.py`:
1. **Loads schema** from `order_schema_v1.avsc` file (for validation)
2. Connects to Schema Registry
3. Fetches the schema using the schema ID from message headers
4. Deserializes and displays the records

The consumer will run for a limited time and then exit.


<div style="background-color: rgba(56, 142, 60, 0.1); border-left: 4px solid #38a169; padding: 12px; margin: 15px 0; border-radius: 4px;">
  <strong>Note:</strong> Kafka consumer groups commit offsets. If you re-run the same group, it may not read old messages again. This lab uses a fresh runtime group ID each time so the consumer can read from the beginning.
</div>

In [ ]:
!python consumer.py

### 🔍 Debug: Check if Messages Exist in the Topic

Use the raw Kafka console consumer to verify messages were actually written to the topic. This bypasses any deserialization issues.

In [ ]:
import subprocess

print("=" * 70)
print("Checking topic 'avro-topic' for messages...")
print("=" * 70)

# Check topic message count
result = subprocess.run(
    """docker exec kafka kafka-console-consumer \
  --bootstrap-server localhost:9092 \
  --topic avro-topic \
  --from-beginning \
  --max-messages 5 \
  --timeout-ms 3000 \
  --property print.key=true""",
    shell=True,
    capture_output=True,
    text=True,
    timeout=10
)

if result.stdout:
    print(f"\n✓ Found messages in topic:")
    print(result.stdout)
else:
    print("\n❌ NO MESSAGES FOUND in topic!")
    print("\nPossible causes:")
    print("  1. Producer failed silently (check producer output above)")
    print("  2. Producer wrote to different topic name")
    print("  3. Messages were written but cleared")
    
if result.stderr:
    print(f"\nDebug info: {result.stderr}")

<div style="background-color: rgba(102, 126, 234, 0.1); border: 1px solid rgba(102, 126, 234, 0.3); padding: 20px; text-align: center; border-radius: 8px; margin-top: 40px;">
  <h3 style="color: #667eea; margin-bottom: 10px;">🎉 Lab 9 Complete!</h3>
  <p style="color: #8b949e; margin: 0;">You've successfully mastered Schema Registry and Avro Serialization! You registered schemas, produced Avro-encoded messages, consumed them with automatic deserialization, and demonstrated backward-compatible schema evolution!</p>
  
  <div style="text-align: left; margin-top: 20px; font-size: 0.95em;">
    <strong>Key Takeaways:</strong>
    <ul>
      <li>✓ Schema Registry provides centralized schema management</li>
      <li>✓ Avro enables compact binary serialization with strong typing</li>
      <li>✓ Schemas are versioned and stored separately</li>
      <li>✓ Schema evolution allows safe schema changes with backward/forward compatibility</li>
      <li>✓ Schema IDs in message headers enable automatic deserialization</li>
      <li>✓ Default values and optional fields enable compatible schema migration</li>
    </ul>
  </div>
</div>